# 10 — Capstone: model versus deterministic and hybrid designs

**Estimated time:** 55 minutes<br>
**Prerequisites:** 09 — Capstone policy-derived ground truth<br>
**Learner-produced evidence:** a policy/model comparison and an explicit architecture choice

## Learning objectives

- Compare a deterministic ceiling with an untouched tiny-model probe.
- Prove that a hybrid renderer cannot alter authoritative decisions.
- Decide which behavior, if any, justifies a language model.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

## Start with the accuracy ceiling

The full deterministic frozen report should be exact for completely
specified rules. The model must therefore justify itself through a
different capability—such as bounded wording—rather than replacing
correct policy decisions with probabilistic ones.


In [ ]:
import json

from aai_local_finetuning.capstone import (
    CAPSTONE_SYSTEM_PROMPT,
    CapstonePrediction,
    build_hybrid_review,
    deterministic_capstone_predictions,
    evaluate_capstone_predictions,
    load_capstone_records,
)
from aai_local_finetuning.modeling import LocalMLXPredictor
from aai_local_finetuning.settings import PROJECT_ROOT, load_settings

settings = load_settings()
source_dir = PROJECT_ROOT / "data" / "processed" / "capstone-readiness-v1"
test_records = load_capstone_records(source_dir / "test.jsonl")
deterministic_report = evaluate_capstone_predictions(
    test_records,
    deterministic_capstone_predictions(test_records),
)
deterministic_report.aggregate.model_dump(mode="json")

## One untouched-model probe

This bounded example shows mechanics, not a winner. The compact model
contract asks for status and non-pass checks. A full model comparison is
expensive and should be run only after methods are locked.


In [ ]:
probe_record = test_records[0]
predictor = LocalMLXPredictor(settings.model_dir)
generated = predictor.generate(
    [
        {"role": "system", "content": CAPSTONE_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": json.dumps(
                probe_record.manifest,
                separators=(",", ":"),
                sort_keys=True,
            ),
        },
    ],
    max_tokens=160,
)
model_prediction = CapstonePrediction(
    example_id=probe_record.example_id,
    raw_text=generated.text,
    latency_ms=generated.latency_ms,
    output_tokens=generated.output_tokens,
    peak_memory_mb=generated.peak_memory_mb,
)
model_probe_report = evaluate_capstone_predictions((probe_record,), (model_prediction,))
{
    "output_preview": generated.text[:500],
    "metrics": model_probe_report.aggregate.model_dump(mode="json"),
    "performance": model_probe_report.performance.model_dump(mode="json"),
}

## The hybrid authority boundary

A renderer receives a frozen check and may return prose only. It has no
channel for changing status, result, severity, rule ID, or remediation ID.
Empty or failing renderers fall back to deterministic policy wording.


In [ ]:
def learner_renderer(check):
    return f"Review note: {check.evidence}"


hybrid = build_hybrid_review(
    probe_record.manifest,
    renderer=learner_renderer,
    renderer_name="notebook_demo",
)
{
    "status_unchanged": (
        hybrid.deterministic_review.status == probe_record.expected_output.status
    ),
    "checks_unchanged": (
        hybrid.deterministic_review.checks == probe_record.expected_output.checks
    ),
    "explanation_preview": hybrid.explanations[0].model_dump(mode="json"),
}

## Optional capstone LoRA smoke

Keep this disabled for Run All. When enabled it writes a notebook-specific
adapter and leaves the canonical capstone change untouched. Falling loss
still does not beat the deterministic ceiling.


In [ ]:
from aai_local_finetuning.training import run_lora

RUN_CAPSTONE_TRAINING = False
if RUN_CAPSTONE_TRAINING:
    capstone_training = run_lora(
        iterations=10,
        config_path=(PROJECT_ROOT / "configs" / "training" / "capstone-lora.yaml"),
        adapter_path=(
            PROJECT_ROOT / "artifacts" / "notebook" / "adapters" / "capstone-smoke"
        ),
        log_name="notebook-capstone-smoke",
    ).model_dump(mode="json")
else:
    capstone_training = {"status": "skipped"}
capstone_training

## Exercise — choose the production shape

Fill in one row per behavior. Success means deterministic checks retain
authority, unavailable facts route outward, and the model is used only
where probabilistic language actually helps.


In [ ]:
architecture_decision = [
    {
        "behavior": "readiness decision",
        "owner": "deterministic policy engine",
        "reason": "exact, auditable rules already define the answer",
    },
    {
        "behavior": "external registry fact",
        "owner": "authorized lookup",
        "reason": "the fact is absent from the local manifest",
    },
    {
        "behavior": "remediation wording",
        "owner": "policy text or constrained tiny-model renderer",
        "reason": "wording may vary without changing authority",
    },
]
architecture_decision

**Hint:** ask what must be correct, what must be looked up, what needs a
person, and what merely benefits from flexible wording.


## Checkpoint

The likely design is deterministic validation plus optional constrained
language generation—not a model pretending to know every readiness fact.

**Next:** `11_design_the_next_project.ipynb` turns the remaining dataset
ideas into review plans without fabricating unverified schemas or rights.
